## Open government data provided by **Canton of Zurich**

_Automatically generated Python starter code for dataset_ **241@statistisches-amt-kanton-zuerich**


## Dataset

# **Anteil 6+ Zi.-Wohnungen &#91;%&#93;**


## Description

Anteil 6+ Zi.-Wohnungen am Wohnungsbestand. Wohnungsbestand gemäss GWS. Wohnungsbestand bis 2009 aus Fortschreibung des Wohnungsbestandes der GWZ aufgrund der WBT.


## Dataset links

[View this dataset in the data catalogue](https://www.zh.ch/de/politik-staat/statistik-daten/datenkatalog.html#/datasets/241%40statistisches-amt-kanton-zuerich)


## Metadata

- **issued:** 2016-01-19T23:00:00
- **modified:** 2026-03-27T12:50:58
- **startDate:** 1990-12-31
- **endDate:** 2024-12-31
- **theme:** &#91;&&#35;x27;http://publications.europa.eu/resource/authority/data-theme/SOCI&&#35;x27;&#93;
- **keyword:** &#91;&&#35;x27;bezirke&&#35;x27;, &&#35;x27;gemeinden&&#35;x27;, &&#35;x27;kanton&#95;zuerich&&#35;x27;, &&#35;x27;nicht&#95;weitergefuehrt&&#35;x27;, &&#35;x27;ogd&&#35;x27;, &&#35;x27;wohnungen&&#35;x27;, &&#35;x27;wohnungsbestand&&#35;x27;, &&#35;x27;zimmerzahl&&#35;x27;&#93;
- **publisher:** &#91;&&#35;x27;Amt für Statistik und Daten&&#35;x27;&#93;
- **landingPage:** https://www.zh.ch/de/politik-staat/gemeinden/gemeindeportraet.html
- **Resource ktzhDistId:** 202
- **Resource title:** Anteil 6+ Zi.-Wohnungen &#91;%&#93;
- **Resource description:** None
- **Resource issued:** 2016-01-20T23:00:00
- **Resource modified:** 2025-09-25T22:00:00


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), and matplotlib (3.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from typing import BinaryIO
from urllib.request import Request, urlopen

USER_AGENT = "OpenZH-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        request = Request(url, headers={"User-Agent": USER_AGENT})
        with urlopen(request, timeout=HTTP_TIMEOUT) as response:
            return get_dataset(response, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **Anteil 6+ Zi.-Wohnungen &#91;%&#93;** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://www.web.statistik.zh.ch/ogd/data/KANTON_ZUERICH_365.csv")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** Amt für Statistik und Daten &#124; Data Shop &#124; datashop@statistik.zh.ch
